# D1-04 First LCA from demand to score in Brightway

This notebook is the main Day 1 milestone: we create the course project, import the bundled BAFU workbook, inspect the resulting database, and run a first LCA.

The hands-on exercises use BAFU throughout the course. A short dedicated section near the end shows how a similar import pattern would look for `ecoinvent`, but we do not run it here because of licensing constraints.


## Learning goals

- Create or switch to the course `brightway` project.
- Import the bundled BAFU workbook with `bw2io.ExcelImporter`.
- Understand the role of strategies, biosphere matching, statistics, and database writing.
- Inspect the imported database by searching for activities and exchanges.
- Run a first LCA score with an explicitly chosen activity and method.
- Recognize how an `ecoinvent` import would fit into the same overall workflow.


## Background references

- Mutel, C. (2017). *Brightway: An open source framework for life cycle assessment*. Journal of Open Source Software, 2(12), 236. https://doi.org/10.21105/joss.00236
- Database of the Swiss Federal Administration, FOEN:20XY, Federal Office for the Environment, 2025.


## 1) Create or switch to the course project

`bw2setup()` creates the core biosphere database and default LCIA methods used by `brightway`.


In [ ]:
from pathlib import Path

import bw2calc as bc
import bw2data as bd
import bw2io as bi

project_name = 'barcelona-rlcia-2026'
bd.projects.set_current(project_name)
print('Current project:', bd.projects.current)

if 'biosphere3' not in bd.databases:
    bi.bw2setup()
    print('Created core biosphere and LCIA methods.')
else:
    print('Core biosphere and LCIA methods already available.')

print('Databases available now:', list(bd.databases))
print('Number of LCIA methods:', len(bd.methods))


## 2) Confirm the bundled BAFU workbook

The workbook used in this course is already distributed with the repository:

- `data/lci-bafu.xlsx`


In [ ]:
bafu_file = Path('data/lci-bafu.xlsx')
print('Workbook path:', bafu_file.resolve())
print('Found:', bafu_file.exists())
if bafu_file.exists():
    print('Workbook size [MB]:', round(bafu_file.stat().st_size / 1024 / 1024, 2))


## 3) Import pipeline: `ExcelImporter` -> strategies -> biosphere matching -> statistics -> database writing

This is the import sequence to understand in this notebook:

1. Create an `ExcelImporter` from the workbook path.
2. Apply the standard import strategies.
3. Match biosphere flows against `biosphere3`.
4. Check import statistics and unlinked exchanges.
5. Write the database to the current project.


In [ ]:
database_name = 'bafu-2025'

if not bafu_file.exists():
    print('Bundled BAFU file not found at data/lci-bafu.xlsx.')
else:
    if database_name in bd.databases:
        print(f"Database '{database_name}' already exists. Skipping import.")
        print('If you want to rerun the full import, use a fresh project or a new database name.')
    else:
        importer = bi.ExcelImporter(str(bafu_file))
        print('Importer type:', type(importer).__name__)
        print('Raw datasets loaded:', len(getattr(importer, 'data', [])))

        importer.apply_strategies()
        print('Applied import strategies.')

        importer.match_database('biosphere3', fields=('name', 'unit', 'categories'))
        print('Matched biosphere exchanges against biosphere3.')

        stats = importer.statistics()
        print('Statistics object:', stats)
        print('Unlinked exchanges after matching:', len(getattr(importer, 'unlinked', [])))

        importer.write_database(database_name)
        print(f"Imported database: {database_name}")

print('Databases now:', list(bd.databases))


## 4) Inspect the imported database

Now we reconnect the import workflow to the Day 1 data model concepts: project -> database -> activity -> exchange.

The search below looks first for `passenger car`, then `electricity`, then `coal`, so that we land on a familiar activity if possible.


In [ ]:
if database_name not in bd.databases:
    print("'bafu-2025' is not available yet. Run the import cell first.")
else:
    db = bd.Database(database_name)
    print('Number of activities in bafu-2025:', len(db))

    search_terms = ['passenger car', 'electricity', 'coal']
    candidate_results = []
    used_term = None
    for term in search_terms:
        hits = db.search(term, limit=5)
        if hits:
            candidate_results = hits
            used_term = term
            break

    print('Search term used:', used_term)
    for act in candidate_results:
        print('-', act['name'], '| location =', act.get('location'), '| unit =', act.get('unit'))


## Checkpoint 1

Pick one activity from the imported database, store it in `selected_activity`, and inspect a few technosphere and biosphere exchanges.

Suggestion: start with a search on `passenger car` or `electricity`.


In [ ]:
# TODO
# Example pattern:
# selected_activity = db.search('passenger car', limit=1)[0]
# print(selected_activity['name'])
# print(list(selected_activity.technosphere())[:3])
# print(list(selected_activity.biosphere())[:3])


In [ ]:
if database_name not in bd.databases:
    print("Run the import cell first.")
else:
    db = bd.Database(database_name)
    candidate_queries = ['passenger car', 'electricity', 'coal']
    selected_activity = None
    for query in candidate_queries:
        hits = db.search(query, limit=1)
        if hits:
            selected_activity = hits[0]
            break

    if selected_activity is None:
        selected_activity = db.random()

    print('Selected activity:', selected_activity['name'])
    print('Location:', selected_activity.get('location'))
    print('Unit:', selected_activity.get('unit'))

    print('First technosphere exchanges:')
    for exc in list(selected_activity.technosphere())[:3]:
        print('  -', exc.input['name'], '| amount =', exc['amount'])

    print('First biosphere exchanges:')
    for exc in list(selected_activity.biosphere())[:3]:
        print('  -', exc.input['name'], '| amount =', exc['amount'])


## Checkpoint 2

Choose one LCIA method that looks like a climate-change or global-warming indicator, and store it in `selected_method`.


In [ ]:
# TODO
# Example pattern:
# [m for m in bd.methods if 'climate change' in ' | '.join(m).lower()][:5]
# selected_method = ...


In [ ]:
preferred_terms = ['climate change', 'global warming', 'gwp']
selected_method = None
for term in preferred_terms:
    hits = [m for m in bd.methods if term in ' | '.join(m).lower()]
    if hits:
        selected_method = hits[0]
        break

if selected_method is None:
    selected_method = next(iter(bd.methods))

print('Selected method:', selected_method)


## 5) Run your first LCA

We now combine the selected activity and method into a complete LCA calculation.


In [ ]:
if database_name not in bd.databases:
    print("'bafu-2025' is not available yet. Run the import cell first.")
elif len(bd.methods) == 0:
    print('No LCIA methods available. Rerun the project-setup cell first.')
else:
    db = bd.Database(database_name)

    if 'selected_activity' not in globals():
        passenger_hits = db.search('passenger car', limit=1)
        selected_activity = passenger_hits[0] if passenger_hits else db.random()

    if 'selected_method' not in globals():
        climate_methods = [m for m in bd.methods if 'climate change' in ' | '.join(m).lower()]
        selected_method = climate_methods[0] if climate_methods else next(iter(bd.methods))

    lca = bc.LCA({selected_activity: 1}, selected_method)
    lca.lci()
    lca.lcia()

    print('Activity:', selected_activity['name'])
    print('Method:', selected_method)
    print('LCA score:', lca.score)


## 6) Dedicated `ecoinvent` import overview

The BAFU workbook is the hands-on database for this course. However, many participants work with `ecoinvent`, so it is useful to see the import pattern.

Key points:
- `ecoinvent` is not distributed in this repository.
- The import is not run during this course.
- The overall logic is the same: create an importer, apply strategies, inspect statistics, then write the database.
- In practice, `ecoinvent` uses dedicated `bw2io` importers.


In [ ]:
# Example only. Do not run in this course.
# from bw2io.importers.ecospold2 import SingleOutputEcospold2Importer
#
# ei_path = '/path/to/unzipped/ecoinvent/datasets'
# ei_name = 'ecoinvent-3.x-cutoff'
#
# ei = SingleOutputEcospold2Importer(ei_path, ei_name)
# ei.apply_strategies()
# ei.statistics()
# ei.write_database()


## 7) Brief troubleshooting notes

- Workbook not found: confirm that `data/lci-bafu.xlsx` exists.
- `biosphere3` missing: rerun the project-setup cell so that `bw2setup()` executes.
- Many unlinked exchanges after import: this is a Day 1 topic again in `D1-06`, where we will debug import problems more systematically.
- Database already exists: use a fresh project or a different database name if you want to repeat the import from scratch.


## Recap

After this notebook, you should now know how to:

- create and activate a `brightway` project
- import a database with `ExcelImporter`
- interpret the role of strategies, matching, and statistics
- search a database for activities and inspect exchanges
- run a first LCA from demand to score
- recognize how the same import logic would extend to `ecoinvent`
